In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'font.size': 8, 'figure.figsize': (10, 5), 'figure.dpi': 120})
COLORS = {'baseline': '#000000', 'prioritized': '#FF6B00'}

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# ============================================
# Configuration - Update these for your runs
# ============================================
WANDB_PROJECT = "verl_priority_sampling"  # Your wandb project name
BASELINE_RUN = 'baseline_run1'            # Baseline run name
PRIORITY_RUN = 'prioritized_run1'         # Priority sampling run name

# ============================================
# Load runs from WandB
# ============================================
api = wandb.Api()
all_runs = {}
for run in api.runs(WANDB_PROJECT):
    df = run.history(samples=5000)
    all_runs[run.name] = {
        'name': run.name,
        'history': df,
        'config': dict(run.config),
    }
    print(f"✅ {run.name}: {len(df)} rows")

print(f"\nTotal: {len(all_runs)} runs")

# ============================================
# Select runs for comparison
# ============================================
baseline = all_runs.get(BASELINE_RUN)
priority = all_runs.get(PRIORITY_RUN)

print(f"\nBaseline: {BASELINE_RUN} - {'✅' if baseline else '❌ not found'}")
print(f"Priority: {PRIORITY_RUN} - {'✅' if priority else '❌ not found'}")

In [ ]:
# ============================================
# Column Finder for Different Metrics
# ============================================

BENCHMARK_PATTERNS = {
    'math500': ['MATH-500', 'math500'],
    'aime24': ['aime', 'AIME', 'aimo'],
}

def find_eval_column(df, benchmark, metric='mean'):
    """Find the eval metric column for a benchmark.
    
    Args:
        df: DataFrame with wandb history
        benchmark: 'math500' or 'aime24'
        metric: 'mean' (pass@1), 'best@2' (pass@2), or 'best@4' (pass@4)
    
    Returns:
        Column name or None
    """
    patterns = BENCHMARK_PATTERNS.get(benchmark, [benchmark])
    
    candidates = []
    for col in df.columns:
        col_lower = col.lower()
        
        if not any(p.lower() in col_lower for p in patterns):
            continue
        if 'val-aux' in col_lower:
            continue
        if col_lower.endswith('/std'):
            continue
            
        if metric == 'mean':
            if 'acc/mean@' in col_lower and 'best@' not in col_lower:
                candidates.append(col)
        elif metric == 'best@4':
            if 'best@4/mean' in col_lower or (col_lower.endswith('best@4') and '/std' not in col_lower):
                candidates.append(col)
        elif metric == 'best@2':
            if 'best@2/mean' in col_lower or (col_lower.endswith('best@2') and '/std' not in col_lower):
                candidates.append(col)
    
    if candidates:
        mean_cols = [c for c in candidates if c.endswith('/mean')]
        return mean_cols[0] if mean_cols else candidates[0]
    return None

In [ ]:
# ============================================
# PLOT: Eval Accuracy (pass@1 and pass@4)
# ============================================
# ⚙️ CONFIGURABLE
PLOT_MIN_STEP = 25     # Start step
PLOT_MAX_STEP = 250   # End step
EVAL_SMOOTH = 3       # Smoothing window (1 = no smoothing)

# Line styles: (metric_key, label, linestyle)
METRICS = [
    ('mean', 'pass@1', '-'),      # solid
    ('best@4', 'pass@4', '--'),   # dashed
]

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(7, 3), sharey=False)
benchmarks = ['math500', 'aime24']

for col, benchmark in enumerate(benchmarks):
    ax = axes[col]
    
    for method_name, run, color in [
        ('Priority', priority, COLORS['prioritized']),
        ('Baseline', baseline, COLORS['baseline']),
    ]:
        if run is None:
            continue
        df = run['history']
        
        for metric_key, metric_label, linestyle in METRICS:
            mean_col = find_eval_column(df, benchmark, metric_key)
            if mean_col is None:
                continue
            
            plot_data = df[['_step', mean_col]].dropna().copy()
            
            # Apply smoothing (backward-only)
            if EVAL_SMOOTH > 1:
                plot_data = plot_data.sort_values('_step')
                plot_data[mean_col] = plot_data[mean_col].rolling(
                    EVAL_SMOOTH, min_periods=1, center=False).mean()
            
            plot_data = plot_data[(plot_data['_step'] >= PLOT_MIN_STEP) & 
                                  (plot_data['_step'] <= PLOT_MAX_STEP)]
            
            label = f"{method_name} {metric_label}"
            ax.plot(plot_data['_step'], plot_data[mean_col], 
                   color=color, linewidth=1.5, linestyle=linestyle, label=label)
    
    ax.set_title(benchmark.upper(), fontweight='bold')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Accuracy')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eval_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Steps {PLOT_MIN_STEP} to {PLOT_MAX_STEP} | Solid=pass@1, Dashed=pass@4")


In [ ]:
# ============================================
# PLOT: Priority Dynamics (priority run only)
# ============================================
# ⚙️ CONFIGURABLE
PLOT_MIN_STEP = 0     # Start step
PLOT_MAX_STEP = 250   # End step
DYNAMICS_SMOOTH = 10  # Smoothing window for noisy metrics
NUM_PROBLEMS = 1000   # Total problems in dataset (for initial heap size)

# Known initial values at step 0 (before any training)
INITIAL_VALUES = {
    'heap_size': NUM_PROBLEMS,
    'solved_pool_size': 0,
    'unsolved_pool_size': 0,
    'std_success_rate': 0,
}

import re

def find_priority_column(df, metric):
    """Find column matching metric name exactly (not as substring)."""
    pattern = re.compile(r'(^|/)' + re.escape(metric) + r'$', re.IGNORECASE)
    matches = [c for c in df.columns if pattern.search(c)]
    return matches[0] if matches else None

if priority:
    df = priority['history']
    
    metrics = [
        ('heap_size', 'Heap Size', 'Count', 1),
        ('solved_pool_size', 'Solved Pool', 'Count', 1),
        ('unsolved_pool_size', 'Unsolved Pool', 'Count', 1),
        ('std_success_rate', 'Std of Success Rates', 'Value', 1),
        ('mean_success_rate', 'Mean Success Rate', 'Value', DYNAMICS_SMOOTH),
        ('mean_priority', 'Mean Priority (ω)', 'Value', DYNAMICS_SMOOTH),
    ]
    
    n_cols = 3
    n_rows = (len(metrics) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7, 2 * n_rows))
    axes = axes.flatten()
    
    for idx, (metric, title, ylabel, smooth) in enumerate(metrics):
        ax = axes[idx]
        is_bottom_row = (idx // n_cols == n_rows - 1)
        
        chosen_col = find_priority_column(df, metric)
        
        if chosen_col:
            plot_data = df[['_step', chosen_col]].dropna().copy()
            
            # Prepend initial value at step 0 if missing
            if metric in INITIAL_VALUES and (len(plot_data) == 0 or plot_data['_step'].min() > 0):
                init_row = pd.DataFrame({'_step': [0], chosen_col: [INITIAL_VALUES[metric]]})
                plot_data = pd.concat([init_row, plot_data], ignore_index=True)
            
            if smooth > 1:
                plot_data = plot_data.sort_values('_step')
                plot_data[chosen_col] = plot_data[chosen_col].rolling(smooth, min_periods=1, center=False).mean()
            
            plot_data = plot_data[(plot_data['_step'] >= PLOT_MIN_STEP) & 
                                  (plot_data['_step'] <= PLOT_MAX_STEP)]
            
            ax.plot(plot_data['_step'], plot_data[chosen_col], 
                   color=COLORS['prioritized'], linewidth=1.5)
            ax.set_ylabel(ylabel)
            ax.set_title(title, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.set_xlabel('Training Step' if is_bottom_row else '')
        else:
            ax.set_title(title, fontweight='bold')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                   transform=ax.transAxes, fontsize=8, color='gray')
    
    for idx in range(len(metrics), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout(h_pad=1.0, w_pad=0.5)
    plt.savefig(FIGURES_DIR / 'priority_dynamics.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No priority run selected")